In [ ]:
# Robust imports and helper definitions for building EEG metadata DataFrame
from typing import Dict, Optional
from pathlib import Path
import json
import pandas as pd
import sys
import h5py
import re, unicodedata, difflib

# true_labels and canonicalization should already be defined in another cell;
# ensure they exist before calling build_eeg_dataframe when running the notebook.

def build_eeg_dataframe(h5_dir: Path, label_map_path: Optional[Path] = None) -> pd.DataFrame:
    """Build EEG metadata dataframe from H5 files.

    - Uses the last underscore in filename to split `subject_id` and `session_id`.
    - Loads an optional `label_map_path` (JSON) mapping labels to indices.
    - Skips malformed files and prints warnings.

    Args:
        h5_dir: Path to directory containing .h5 files (Path or str accepted)
        label_map_path: optional path to a JSON label->index map
    Returns:
        pd.DataFrame with rows for every epoch (subject_id, session_id, epoch_idx, label_name, label_idx, n_channels, n_samples, fs, path_h5)
    """
    h5_dir = Path(h5_dir)

    # Load label map if provided
    label2idx: Dict[str, int] = {}
    if label_map_path:
        try:
            with open(label_map_path, 'r', encoding='utf-8') as f:
                label2idx = json.load(f)
            print(f"Loaded label map with {len(label2idx)} entries")
        except FileNotFoundError:
            print(f"Label map not found at {label_map_path}; continuing without it")
        except Exception as e:
            print(f"Warning reading label map: {e}; continuing without it")

    rows = []
    skipped = []

    h5_files = sorted(h5_dir.glob('*.h5'))
    if not h5_files:
        print(f"⚠ No H5 files found in {h5_dir}")
        return pd.DataFrame()

    for file in h5_files:
        stem = file.stem
        if '_' not in stem:
            skipped.append(str(file))
            print(f"Skipping file with unexpected name (no underscore): {file.name}")
            continue

        # Use the last underscore as separator to support subject names that contain underscores
        subject_id, session_id = stem.rsplit('_', 1)

        try:
            with h5py.File(file, 'r') as f:
                if 'data' not in f or 'labels' not in f:
                    print(f"Warning: expected datasets missing in {file.name}")
                    skipped.append(str(file))
                    continue

                data = f['data']
                labels = f['labels'][:]  # type: ignore
                n_epochs, n_channels, n_samples = data.shape  # type: ignore

                for i, lbl_raw in enumerate(labels):  # type: ignore
                    # A canonicalize_label function is expected elsewhere in the notebook
                    try:
                        label_name = canonicalize_label(lbl_raw)
                    except Exception:
                        # Fallback: try simple decode
                        label_name = lbl_raw.decode('utf-8', 'ignore') if isinstance(lbl_raw, (bytes, bytearray)) else str(lbl_raw)

                    label_idx = label2idx.get(label_name, -1)

                    rows.append({
                        'subject_id': str(subject_id),
                        'session_id': str(session_id),
                        'epoch_idx': int(i),
                        'label_name': label_name,
                        'label_idx': int(label_idx) if isinstance(label_idx, (int, float)) else label_idx,
                        'n_channels': int(n_channels),
                        'n_samples': int(n_samples),
                        'fs': 256,
                        'path_h5': str(file)
                    })

            print(f"  ✓ Processed {file.name}: {n_epochs} epochs")
        except Exception as e:
            print(f"  ✗ Error processing {file.name}: {e}")
            skipped.append(str(file))
            continue

    df = pd.DataFrame(rows)
    if skipped:
        print(f"Skipped {len(skipped)} files; examples: {[Path(s).name for s in skipped[:10]]}")
    return df


## CONFIG

In [14]:
# Path configuration function with user-specific bypass
from typing import Dict, List, Tuple
from pathlib import Path
import json
import pandas as pd
import sys

# Prefer an explicit project_root when running from a notebook
project_root = Path.cwd().parents[0]  # adjust if your notebook starts elsewhere
meta_csv = project_root / "data" / "interim" / "eeg_metadata.csv"
out = project_root / "data" / "interim" / "label2idx.json"

def get_data_paths(user_name: str = None) -> Dict[str, Path]:
    """
    Get data paths with user-specific overrides.
    
    The OneDrive path contains ALL subjects' H5 files:
    - Files are named as {subject_id}_{session_id}.h5
    - Example: 00_01.h5, 00_02.h5, ..., 01_01.h5, 01_02.h5, etc.
    - 5 acquisitions per subject (e.g., 00_01 to 00_05 for subject 00)
    
    For user 'daniele', uses OneDrive path (contains all subjects' data).
    For other users, uses default project paths.
    
    Args:
        user_name: Name of the user (e.g., 'daniele') - NOT the subject ID
    
    Returns:
        Dictionary with 'h5_dir' and 'eloc_path'
    """
    paths = {}
    
    # Electrode locations file - always in src/io
    paths['eloc_path'] = project_root / "src" / "io" / "ebneuro.elocs"
    
    # H5 data files path - user-specific bypass
    if user_name and user_name.lower() == 'daniele':
        # Bypass for user daniele: use OneDrive path (contains ALL subjects' data)
        # All H5 files (00_01.h5, 00_02.h5, 01_01.h5, etc.) are in this directory
        paths['h5_dir'] = Path('/Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - h5/data')
        print(f"✓ Using daniele's OneDrive path (all subjects): {paths['h5_dir']}")
    else:
        # Default path for other users
        paths['h5_dir'] = project_root / "data" / "processed"
        print(f"✓ Using default path: {paths['h5_dir']}")
    
    # Verify paths exist
    if not paths['eloc_path'].exists():
        print(f"⚠ Warning: Electrode file not found at {paths['eloc_path']}")
    if not paths['h5_dir'].exists():
        print(f"⚠ Warning: H5 directory not found at {paths['h5_dir']}")
    
    return paths

# Configure paths - CHANGE THIS TO YOUR USER NAME
USER_NAME = 'daniele'  # Set to 'daniele' to use OneDrive path (all subjects), or None for default
paths = get_data_paths(USER_NAME)

print(f"\n📁 Configuration:")
print(f"   Electrode file: {paths['eloc_path']}")
print(f"   H5 data dir: {paths['h5_dir']}")
print(f"   Expected files: 00_01.h5, 00_02.h5, ..., 01_01.h5, etc.")

✓ Using daniele's OneDrive path (all subjects): /Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - h5/data
⚠ Warning: Electrode file not found at /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/src/io/ebneuro.elocs

📁 Configuration:
   Electrode file: /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/src/io/ebneuro.elocs
   H5 data dir: /Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - h5/data
   Expected files: 00_01.h5, 00_02.h5, ..., 01_01.h5, etc.


In [15]:
import re, unicodedata, difflib
import h5py
import pandas as pd
from pathlib import Path
import json

## List of true labels in Italian
true_labels = [
    "arrivare","andare","aspettare","avere","capire","chiamare","chiedere","conoscere","dare","dire","dovere",
    "essere","fare","mettere","potere","prendere","sapere","sentire","trovare","venire","aprire","chiudere",
    "mangiare","bere","accendere","spegnere","volere","bene","si","no","più","poco","molto","sempre","adesso",
    "poi","male","sopra","sotto","destra","sinistra","avanti","indietro","oggi","domani","ieri","forse","prima",
    "perchè","anche","come","però","quindi","quando","dove","se","oppure","io","lui","lei","noi","voi","loro",
    "tu","questo","quello","buono","bello","cattivo","brutto","grande","piccolo","nuovo","vecchio","cosa","parte",
    "anno","casa","problema","aiuto","tempo","lavoro","persona","acqua","cibo","bisogno","donna","uomo","gruppo",
    "guerra","idea","macchina","mano","oggetto","telefono","computer","domanda","uno","mille","paura","ansia",
    "gioia","felicità","tristezza","serenità","amore","morte","bagno","dolore","riposo"
]

def _strip_diacritics(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def _robust_decode(x) -> str:
    if isinstance(x, (bytes, bytearray)):
        for enc in ('utf-8', 'latin-1', 'cp1252'):
            try:
                s = x.decode(enc)
                break
            except Exception:
                continue
    else:
        s = str(x)

    s = s.replace('\x00', '').strip()

    if ('Ã' in s) or ('Â' in s):
        try:
            s = s.encode('latin-1', 'ignore').decode('utf-8', 'ignore')
        except Exception:
            pass

    s = unicodedata.normalize('NFC', s)
    return s

def canonicalize_label(raw) -> str:
    s = _robust_decode(raw).lower()

    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'_img$', '', s)

    s = (s.replace('’', "'")
           .replace('‘', "'")
           .replace('“', '"')
           .replace('”', '"'))
    s = s.replace("''", "'").replace("´", "'").replace("`", "'").strip()

    fixes = {
        "perche": "perchè", "perche'": "perchè", "perch'e": "perchè", "perchà''": "perchè",
        "pero": "però", "piu": "più", "serenita": "serenità", "felicita": "felicità"
    }
    if s in fixes: 
        return fixes[s]

    if s in true_labels:
        return s

    s_ascii = _strip_diacritics(s)
    candidates_ascii = [_strip_diacritics(t) for t in true_labels]

    match1 = difflib.get_close_matches(s_ascii, candidates_ascii, n=1, cutoff=0.7)
    if match1:
        idx = candidates_ascii.index(match1[0])
        return true_labels[idx]

    match2 = difflib.get_close_matches(s, true_labels, n=1, cutoff=0.6)
    if match2:
        return match2[0]

    return s


In [16]:

def build_eeg_dataframe(h5_dir, label_map_path):
    with open(label_map_path, "r", encoding="utf-8") as f:
        label2idx = json.load(f)

    rows = []
    for file in sorted(Path(h5_dir).glob("*.h5")):
        subject_id, session_id = file.stem.split("_")
        with h5py.File(file, "r") as f:
            data = f["data"]
            labels = f["labels"][:] # type: ignore
            n_epochs, n_channels, n_samples = data.shape # type: ignore

            for i, lbl_raw in enumerate(labels): # type: ignore
                label_name = canonicalize_label(lbl_raw)
                label_idx = label2idx.get(label_name, -1)
                rows.append({
                    "subject_id": subject_id,
                    "session_id": session_id,
                    "epoch_idx": i,
                    "label_name": label_name,
                    "label_idx": label_idx,
                    "n_channels": n_channels,
                    "n_samples": n_samples,
                    "fs": 256,
                    "path_h5": str(file)
                })

    df = pd.DataFrame(rows)
    return df


## 🧠 EEG Intelligent DataFrame – Overview

This DataFrame acts as the **central index** for your entire EEG dataset.  
Instead of duplicating raw data, it keeps **structured metadata** that describes every EEG epoch — including where it’s stored, which subject/session it belongs to, and its semantic label.

---

### 📁 Structure

Each row of the DataFrame corresponds to **one EEG epoch** and contains:

| Column | Description |
|:--------|:-------------|
| `subject_id` | ID of the participant (e.g. `11`) |
| `session_id` | Recording session (e.g. `S002`) |
| `epoch_idx` | Index of the epoch inside the `.h5` file |
| `label_name` | Semantic label (e.g. *"felicità"*, *"paura"*) |
| `label_idx` | Numerical class index from `label2idx.json` |
| `n_channels` | Number of EEG channels in that epoch |
| `n_samples` | Number of samples per channel |
| `path_h5` | Absolute path to the `.h5` file containing the signal |

---

### ⚙️ Purpose

The **EEG Intelligent DataFrame** serves as a lightweight, queryable "map" of your dataset.  
It allows you to:

1. **Explore** dataset composition and class balance.  
2. **Filter and retrieve** EEG signals on demand (by subject, session, or label).  
3. **Attach new features** (e.g., spectral power, connectivity metrics).  
4. **Build higher-level datasets** for PyTorch or PyTorch Geometric (graphs).  
5. **Visualize** or debug specific epochs without reloading everything.



In [18]:
try:
    print("Reading:", meta_csv)
    df = pd.read_csv(meta_csv)
except FileNotFoundError:
    print(f"ERROR: metadata CSV not found at {meta_csv}")
    sys.exit(1)
except Exception as e:
    print("ERROR reading CSV:", repr(e))
    sys.exit(1)

if "label_name" not in df.columns:
    print("ERROR: 'label_name' column not found. Available columns:", df.columns.tolist())
    # optionally try to guess a label column, e.g. 'label' or 'label_raw'
else:
    labels = sorted(df["label_name"].dropna().unique())
    label2idx = {label: idx for idx, label in enumerate(labels)}
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(label2idx, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved {len(label2idx)} labels to {out}")

Reading: /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/interim/eeg_metadata.csv
ERROR reading CSV: EmptyDataError('No columns to parse from file')


SystemExit: 1

In [21]:

h5_dir = paths['h5_dir']
label_map_path = project_root / "data" / "interim" / "label2idx.json"
output_csv = project_root / "data" / "interim" / "eeg_metadata.csv"

meta_df_eeg = build_eeg_dataframe(
    h5_dir=str(h5_dir),
    label_map_path=str(label_map_path)
)

meta_df_eeg.to_csv(output_csv, index=False)

meta_df_eeg

ValueError: too many values to unpack (expected 2)

In [12]:
# Number of occurrences of each label
print(meta_df_eeg["label_name"].value_counts().head(10))


KeyError: 'label_name'

In [28]:
# Number of epochs per subject
print(meta_df_eeg.groupby("subject_id")["epoch_idx"].count())


subject_id
11    1094
Name: epoch_idx, dtype: int64


In [29]:
# Classes present in each subject
print(meta_df_eeg.groupby("subject_id")["label_name"].nunique())


subject_id
11    110
Name: label_name, dtype: int64


In [30]:
# Percentage of each class
print(meta_df_eeg["label_name"].value_counts(normalize=True) * 100)


label_name
cibo        0.914077
acqua       0.914077
spegnere    0.914077
dolore      0.914077
trovare     0.914077
              ...   
se          0.914077
prima       0.914077
uno         0.731261
si          0.731261
telefono    0.731261
Name: proportion, Length: 110, dtype: float64


In [22]:
# Each epoch of subject 11 with class "felicità"
subset = meta_df_eeg.query("subject_id == '11' and label_name == 'felicità'")
print(subset)

     subject_id session_id  epoch_idx label_name  label_idx  n_channels  \
105          11       S001        105   felicità        102          61   
215          11       S001        215   felicità        102          61   
246          11       S002         26   felicità        102          61   
353          11       S002        133   felicità        102          61   
530          11       S003         96   felicità        102          61   
640          11       S003        206   felicità        102          61   
750          11       S004         96   felicità        102          61   
860          11       S004        206   felicità        102          61   
949          11       S005         75   felicità        102          61   
1059         11       S005        185   felicità        102          61   

      n_samples   fs                                            path_h5  
105         384  256  /Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...  
215         384  256  /Use

## EEG Spectral Feature Extraction

For each EEG epoch (1.5 s, 256 Hz sampling rate), spectral features were extracted using **Welch’s Power Spectral Density (PSD)** method.  
The PSD represents how signal power is distributed across frequencies, measured in **µV²/Hz**, since the EEG amplitude was originally expressed in microvolts.

The PSD was integrated within canonical EEG frequency bands to obtain **band power values** in **µV²**:

| Band | Frequency Range (Hz) | Description |
|:-----|:---------------------|:-------------|
| **Delta (δ)** | 1 – 4 | Slow-wave activity, associated with deep sleep or low vigilance |
| **Theta (θ)** | 4 – 8 | Memory processes, drowsiness, limbic activation |
| **Alpha (α)** | 8 – 13 | Relaxed wakefulness, visual idling, eyes-closed resting state |
| **Beta (β)** | 13 – 30 | Motor activity, alertness, active cognitive processing |
| **Gamma (γ)** | 30 – 45 | Fast oscillations, sensory binding, high-level cognition |

### Computed Metrics

Each epoch is described by the following metrics:

| Metric | Definition | Unit | Description |
|:--------|:------------|:------|:-------------|
| **`total_power`** | \(\displaystyle P_\text{tot} = \int_{1}^{45} PSD(f)\,df\) | µV² | Total signal power across 1–45 Hz |
| **`delta`, `theta`, `alpha`, `beta`, `gamma`** | Band power from integration within each band | µV² | Absolute power per band |
| **`*_rel`** | \(\displaystyle P_\text{band} / P_\text{tot}\) | dimensionless | Relative contribution of each band to total power |
| **`alpha_beta_ratio`** | \(\displaystyle \frac{P_\alpha}{P_\beta}\) | dimensionless | Indicator of relaxation vs. activation |
| **`theta_alpha_ratio`** | \(\displaystyle \frac{P_\theta}{P_\alpha}\) | dimensionless | Cognitive fatigue or attentional engagement index |

The relative power values (`*_rel`) sum approximately to 1, representing the normalized spectral composition of each epoch.

### Physiological Interpretation

- **High delta/theta** → low vigilance or drowsy states  
- **High alpha** → relaxed or eyes-closed resting condition  
- **High beta** → active engagement or motor planning  
- **High gamma** → fast cognitive or perceptual integration  
- **Alpha/Beta ratio** → higher in calm or relaxed conditions  
- **Theta/Alpha ratio** → higher in fatigue or stress

---


In [6]:
df_band=pd.read_csv("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/interim/bandpowers.csv")
df_band.tail()

,subject_id,session_id,epoch_idx,channel,label_name,total_power,delta,theta,alpha,beta,gamma,delta_rel,theta_rel,alpha_rel,beta_rel,gamma_rel,alpha_beta_ratio,theta_alpha_ratio
2389975,[8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8...,8,109,EEG57,donna_img,16.173124,3.246977,3.365572,7.954331,1.042439,0.475601,0.200764,0.208097,0.491824,0.064455,0.029407,7.630502,0.423112
2389976,[8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8...,8,109,EEG58,donna_img,8.814912,2.601838,1.976928,3.185019,0.477384,0.471945,0.295163,0.224271,0.361322,0.054156,0.053539,6.671822,0.620696
2389977,[8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8...,8,109,EEG59,donna_img,6.729763,3.305472,1.171036,1.284515,0.464466,0.455159,0.491172,0.174009,0.190871,0.069017,0.067634,2.765570,0.911657
2389978,[8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8...,8,109,EEG60,donna_img,6.684262,1.533306,1.081791,1.943519,1.147507,0.908704,0.229390,0.161841,0.290760,0.171673,0.135947,1.693688,0.556614
2389979,[8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8...,8,109,EEG61,donna_img,6.960878,1.473981,1.133689,2.045619,1.217615,1.022227,0.211752,0.162866,0.293874,0.174923,0.146853,1.680021,0.554203


In [7]:
def build_channel_mapping(h5_path: Path, eloc_path: Path):
    with h5py.File(h5_path, "r") as f:
        n_channels = f["data"].shape[1] # type: ignore
    h5_channels = [f"EEG{i+1}" for i in range(n_channels)]

    df_eloc = pd.read_csv(eloc_path, sep=r"\s+", header=None, engine="python")
    eloc_channels = df_eloc.iloc[:, -1].astype(str).tolist()

    n_min = min(len(h5_channels), len(eloc_channels))
    mapping = pd.DataFrame({
        "channel_index": range(1, n_min + 1),
        "h5_name": h5_channels[:n_min],
        "eloc_name": eloc_channels[:n_min]
    })
    return mapping

In [15]:
eloc_path = "/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/src/io/ebneuro.locs"
h5_path = paths['h5_dir'] / "00_01.h5"
mapping_df = build_channel_mapping(h5_path, eloc_path)
mapping_df.to_csv(project_root / "data" / "interim" / "channel_mapping.csv", index=False)

mapping_df


,channel_index,h5_name,eloc_name
0,1,EEG1,A1
1,2,EEG2,AF7
2,3,EEG3,AF3
3,4,EEG4,Fp1
4,5,EEG5,FP2
...,...,...,...
56,57,EEG57,AFz
57,58,EEG58,Fz
58,59,EEG59,FCz
59,60,EEG60,Cz
